# Link Prediction -- TransE

Builds a pykeen TriplesFactory from the LP subgraph tensors, trains via a
manual training loop with literature-anchored hyperparameters (only
`embedding_dim` is tuned, via `DIM_GRID`). Evaluated with the same
`evaluate_link_prediction` ranking function as the RGCN/GraphSAGE LP
notebooks, for directly comparable MRR/Hits@k.

**Run `link_prediction_setup.ipynb` first.**

In [ ]:
%restart_python

In [ ]:
# Mitigates CUDA memory fragmentation, and (more importantly here) clears
# any GPU memory a PREVIOUS kernel/process on this cluster left allocated --
# a real, reproduced OOM in link_prediction_rgcn.ipynb turned out to be
# exactly this: the GPU already had 15.45/15.56 GiB in use by a leftover
# process before this notebook's own first tensor.to(device) call even ran.
# Must run before any CUDA context is created, right after the restart above.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import os
import sys

os.environ['PYTORCH_JIT'] = '0'
repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import time
import numpy as np
import torch
import pandas as pd

from pykeen.triples import TriplesFactory
from pykeen.models import TransE

from node_prediction.gnn_tools import cleanup
from link_prediction.link_prediction_tools import evaluate_link_prediction, quick_sanity_check, sample_negatives

In [ ]:
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
LP_DIR = f"{volume}link_prediction/"

# margin/scoring_fct_norm match the original TransE paper. optimizer_cls is Adam,
# not the paper's SGD -- SGD's loss crawled ~0.0001/epoch on this 2.6M-entity
# subgraph (170x FB15k's scale); Adam at the same lr dropped it 0.50->0.24 in 10
# epochs. neg_per_pos=4 kept for consistency with link_prediction_rgcn/graphsage.ipynb.
# embedding_dim is swept over DIM_GRID (not literature-fixed), winner picked by val MRR.
DIM_GRID = [32, 64]
LP_HPARAMS = {
    'lr': 0.01,
    'margin': 0.5,
    'scoring_fct_norm': 2,
    'optimizer_cls': torch.optim.Adam,
    'neg_per_pos': 4,
    'epochs': 100,  # max epoch budget -- early stopping normally ends a run well before this
    'patience': 15,  # in EPOCHS, not checks -- same unit convention as gnn_tools.train_model / train_link_prediction_model
    'eval_frequency': 5,  # validate every 5 epochs; each check re-runs the val-MRR ranking eval, so this trades check frequency for overhead
}

# Caps periodic in-training val-eval cost; final test eval is always uncapped.
MAX_VAL_EVAL_TRIPLES = 1000

# 'macro' (mean of each relation's own MRR) avoids 'pooled' being dominated
# by large-candidate-pool relations. Matches link_prediction_rgcn.ipynb.
MONITOR = 'macro'

# Run this notebook once per schema, in separate kernel sessions -- see
# link_prediction_setup.ipynb's SCHEMA_LABEL convention.
SCHEMA_LABEL = 'sulo'  # <<< CHANGE TO 'base' FOR THE SECOND RUN


def build_pykeen_factory_from_subgraph(sub_data, train_triples_by_relation, role_chain_relation_names):
    """Builds a pykeen TriplesFactory from the subgraph's real edges, plus
    explicit synthetic triples for ROLE_CHAIN relations (not present as real
    edges). Returns (factory, pykeen_relation_fwd_ids) -- the latter maps each
    ROLE_CHAIN relation to its assigned pykeen id; DIRECT relations' ids are
    derived separately by the caller as real edge_type // 2."""
    edge_index = sub_data.edge_index
    edge_type = sub_data.edge_type
    fwd_mask = (edge_type % 2 == 0)
    h = edge_index[0, fwd_mask]
    t = edge_index[1, fwd_mask]
    r = edge_type[fwd_mask] // 2  # undo the rel*2 doubling -- pykeen wants plain relation ids

    num_real_relations = int(r.max().item()) + 1 if r.numel() > 0 else 1

    extra_h, extra_t, extra_r = [], [], []
    pykeen_relation_fwd_ids = {}
    next_id = num_real_relations
    for relation_name in train_triples_by_relation:
        if relation_name not in role_chain_relation_names:
            continue
        triples = train_triples_by_relation[relation_name]
        if triples.size(0) > 0:
            extra_h.append(triples[:, 0])
            extra_t.append(triples[:, 1])
            extra_r.append(torch.full((triples.size(0),), next_id, dtype=torch.long))
        pykeen_relation_fwd_ids[relation_name] = next_id
        next_id += 1

    if extra_h:
        h = torch.cat([h] + extra_h)
        t = torch.cat([t] + extra_t)
        r = torch.cat([r] + extra_r)

    mapped_triples = torch.stack([h, r, t], dim=1)
    entity_to_id = {str(i): i for i in range(sub_data.num_nodes)}
    relation_to_id = {str(i): i for i in range(next_id)}

    factory = TriplesFactory(
        mapped_triples=mapped_triples,
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=False,
    )
    return factory, pykeen_relation_fwd_ids


def make_transe_score_fn(model, device=None):
    """Manual TransE scoring (-||h+r-t||), not pykeen's own evaluator, so
    this is scored by the same evaluate_link_prediction() code RGCN/GraphSAGE
    use. r_ids must be valid pykeen relation ids for this model (see
    pykeen_relation_fwd_ids)."""
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device).eval()
    entity_emb = model.entity_representations[0]().detach()
    relation_emb = model.relation_representations[0]().detach()

    def score_fn(h_ids, r_ids, t_ids):
        with torch.no_grad():
            h_e = entity_emb[h_ids.to(device)]
            t_e = entity_emb[t_ids.to(device)]
            r_e = relation_emb[r_ids.to(device)]
            return -torch.norm(h_e + r_e - t_e, p=2, dim=-1).cpu()

    return score_fn


def macro_mrr(metrics):
    """Unweighted mean of each relation's own MRR, excluding 'pooled' and
    any non-'ok' relation. Local copy of link_prediction_tools' macro_mrr
    since TransE's training loop has no shared call site to hang it off."""
    vals = [m['mrr'] for name, m in metrics.items() if name != 'pooled' and m.get('status') == 'ok']
    return float(np.mean(vals)) if vals else float('nan')


def train_transe_on_subgraph(factory, emb_dim, lr, epochs,
                              train_triples_by_relation, val_triples_by_relation,
                              candidate_pools, true_tails_by_relation, relation_fwd_ids,
                              margin=0.5, scoring_fct_norm=2, optimizer_cls=torch.optim.Adam,
                              neg_per_pos=4, batch_size=1024, seed=42, patience=15,
                              eval_frequency=5, monitor='macro', max_val_eval_triples=None):
    """Trains TransE via a manual per-relation mini-batch loop (not
    pykeen's SLCWATrainingLoop) with tail-only, type-constrained negative
    sampling via sample_negatives(candidate_pools) -- matching
    evaluate_link_prediction's own convention. This matters: a uniform-
    random global negative sampler (pykeen's default) trains against a much
    easier task than the type-constrained ranking eval measures, producing
    misleadingly poor/inverted AUC despite normal-looking train loss.

    optimizer_cls defaults to Adam, not the original paper's SGD -- SGD's
    loss crawled ~0.0001/epoch on this 2.6M-entity subgraph.

    factory is used only to size the model's embedding tables; training
    iterates train_triples_by_relation directly, not factory.mapped_triples.

    Returns (model, best_val_mrr, best_val_mrr_pooled, best_val_mrr_macro,
    best_epoch, history), with best-val-MRR weights already loaded into model.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  [train_transe_on_subgraph] device={device} | num_entities={factory.num_entities} "
          f"| num_relations={factory.num_relations} | emb_dim={emb_dim}")

    eval_val_triples_by_relation = val_triples_by_relation
    if max_val_eval_triples is not None:
        eval_val_triples_by_relation = {}
        _val_sample_gen = torch.Generator().manual_seed(seed)
        for rel_name, triples in val_triples_by_relation.items():
            if triples is not None and triples.size(0) > max_val_eval_triples:
                keep_idx = torch.randperm(triples.size(0), generator=_val_sample_gen)[:max_val_eval_triples]
                eval_val_triples_by_relation[rel_name] = triples[keep_idx]
            else:
                eval_val_triples_by_relation[rel_name] = triples

    def build_model():
        return TransE(
            triples_factory=factory, embedding_dim=emb_dim, scoring_fct_norm=scoring_fct_norm,
            random_seed=seed,
        ).to(device)

    model = build_model()
    optimizer = optimizer_cls(params=model.get_grad_params(), lr=lr)
    loss_fn = torch.nn.MarginRankingLoss(margin=margin)

    train_by_pykeen_rid = {}
    for rel_name, triples in train_triples_by_relation.items():
        if triples is None or triples.size(0) == 0:
            continue
        train_by_pykeen_rid[relation_fwd_ids[rel_name]] = (rel_name, triples)

    best_val_mrr = -1.0
    best_val_mrr_pooled = float('nan')
    best_val_mrr_macro = float('nan')
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss, epoch_batches = 0.0, 0
        for pykeen_r_id, (rel_name, triples) in train_by_pykeen_rid.items():
            perm = torch.randperm(triples.size(0))
            for start in range(0, triples.size(0), batch_size):
                batch = triples[perm[start:start + batch_size]]
                h, t = batch[:, 0], batch[:, 1]

                neg_t = sample_negatives(h, candidate_pools[rel_name], true_tails_by_relation[rel_name], k=neg_per_pos)
                h_rep = h.repeat_interleave(neg_per_pos)
                r_col = torch.full((h.size(0),), pykeen_r_id, dtype=torch.long)
                r_col_rep = torch.full((h_rep.size(0),), pykeen_r_id, dtype=torch.long)

                pos_hrt = torch.stack([h, r_col, t], dim=1).to(device)
                neg_hrt = torch.stack([h_rep, r_col_rep, neg_t], dim=1).to(device)

                optimizer.zero_grad()
                pos_scores = model.score_hrt(pos_hrt).squeeze(-1)
                neg_scores = model.score_hrt(neg_hrt).squeeze(-1)
                # repeat_interleave aligns pos/neg 1:1 for MarginRankingLoss.
                pos_scores_rep = pos_scores.repeat_interleave(neg_per_pos)
                loss = loss_fn(pos_scores_rep, neg_scores, torch.ones_like(pos_scores_rep))
                loss.backward()
                # Same gradient-clip guard used for RGCN/GraphSAGE training.
                torch.nn.utils.clip_grad_norm_(model.get_grad_params(), max_norm=1.0)
                optimizer.step()
                epoch_loss += loss.item()
                epoch_batches += 1

        avg_loss = epoch_loss / max(epoch_batches, 1)

        if epoch % eval_frequency == 0 or epoch == epochs:
            score_fn = make_transe_score_fn(model)
            val_metrics = evaluate_link_prediction(
                score_fn, eval_val_triples_by_relation, candidate_pools,
                true_tails_by_relation, relation_fwd_ids,
            )
            val_mrr_pooled = val_metrics['pooled']['mrr']
            val_mrr_macro_ = macro_mrr(val_metrics)
            selection_mrr = val_mrr_macro_ if monitor == 'macro' and not np.isnan(val_mrr_macro_) else val_mrr_pooled

            record = {'epoch': epoch, 'train_loss': avg_loss,
                      'val_mrr': val_mrr_pooled, 'val_mrr_macro': val_mrr_macro_}
            for rel_name, m in val_metrics.items():
                if rel_name == 'pooled' or m.get('status') != 'ok':
                    continue
                record[f'val_mrr_{rel_name}'] = m['mrr']
            history.append(record)

            print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | "
                  f"Val MRR pooled={val_mrr_pooled:.4f} macro={val_mrr_macro_:.4f} (monitor={monitor})")

            if selection_mrr > best_val_mrr:
                best_val_mrr = selection_mrr
                best_val_mrr_pooled = val_mrr_pooled
                best_val_mrr_macro = val_mrr_macro_
                best_epoch = epoch
                epochs_no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += eval_frequency

            del score_fn
            cleanup()

            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} "
                      f"(no Val MRR improvement in {patience} epochs).")
                break

    final_model = build_model()
    if best_state is not None:
        final_model.load_state_dict(best_state)

    return final_model, best_val_mrr, best_val_mrr_pooled, best_val_mrr_macro, best_epoch, history


## Run the small validation search (embedding_dim only) + final test evaluation

In [ ]:
lp_data = torch.load(f"{LP_DIR}{SCHEMA_LABEL}_lp_data.pt", weights_only=False)
sub_graph_data = lp_data['sub_graph_data']
role_chain_relation_names = lp_data['role_chain_relation_names']
print(f"[{SCHEMA_LABEL}] sub_graph_data: num_nodes={sub_graph_data.num_nodes} "
      f"num_edges={sub_graph_data.edge_index.size(1)}")

# Timed explicitly: with num_nodes in the millions, entity_to_id's dict
# comprehension + pykeen's own TriplesFactory bookkeeping can itself take a
# long time, BEFORE any actual training starts -- this print tells you
# whether a stall is happening here or inside train_transe_on_subgraph.
_t_factory0 = time.time()
train_factory, role_chain_pykeen_ids = build_pykeen_factory_from_subgraph(
    sub_graph_data, lp_data['train_triples_by_relation'], role_chain_relation_names,
)
print(f"[{SCHEMA_LABEL}] TriplesFactory built in {time.time() - _t_factory0:.1f}s "
      f"| num_entities={train_factory.num_entities} num_triples={train_factory.num_triples}")

# DIRECT relations: pykeen id = the bundle's graph-side doubled edge_type
# id // 2 (matches r = edge_type[fwd_mask] // 2 above). ROLE_CHAIN
# relations: the fresh id build_pykeen_factory_from_subgraph assigned --
# this is a purely LOCAL numbering for this TransE model, independent of
# the RGCN/GraphSAGE pipelines' own (also purely local) relation ids.
pykeen_relation_fwd_ids = dict(role_chain_pykeen_ids)
for relation_name, fwd_id in lp_data['relation_fwd_ids'].items():
    if relation_name not in role_chain_relation_names:
        pykeen_relation_fwd_ids[relation_name] = fwd_id // 2

print(f"[{SCHEMA_LABEL}] Fixed hyperparameters (literature-anchored, emb_dim swept "
      f"over {DIM_GRID}): {LP_HPARAMS}")

# Small validation search over DIM_GRID only -- everything else stays at the
# literature-fixed LP_HPARAMS above. Mirrors
# link_prediction_tools.run_small_lp_validation_search's structure (used by
# RGCN/GraphSAGE), but doesn't need that function's checkpoint/resume/OOM-
# retry machinery: TransE trains fast, and now early-stops on val MRR the
# same way RGCN/GraphSAGE do (see train_transe_on_subgraph), so each dim is
# one train-with-early-stopping-then-evaluate-on-val pass.
search_rows = []
best_val_mrr, best_dim, best_model, best_history = -1.0, None, None, None
for dim in DIM_GRID:
    print(f"\n[LP VALIDATION SEARCH] TransE/{SCHEMA_LABEL} | emb_dim={dim}")
    t0 = time.time()
    model, val_mrr_selected, val_mrr_pooled, val_mrr_macro, best_epoch_this_dim, train_history = train_transe_on_subgraph(
        train_factory, emb_dim=dim, lr=LP_HPARAMS['lr'], epochs=LP_HPARAMS['epochs'],
        train_triples_by_relation=lp_data['train_triples_by_relation'],
        val_triples_by_relation=lp_data['val_triples_by_relation'],
        candidate_pools=lp_data['candidate_pools'],
        true_tails_by_relation=lp_data['true_tails_by_relation'],
        relation_fwd_ids=pykeen_relation_fwd_ids,
        margin=LP_HPARAMS['margin'], scoring_fct_norm=LP_HPARAMS['scoring_fct_norm'],
        optimizer_cls=LP_HPARAMS['optimizer_cls'], neg_per_pos=LP_HPARAMS['neg_per_pos'],
        patience=LP_HPARAMS['patience'], eval_frequency=LP_HPARAMS['eval_frequency'], monitor=MONITOR,
        max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
    )
    elapsed = time.time() - t0
    # val_mrr_pooled/val_mrr_macro are already the figures AT the restored
    # best-val-MRR checkpoint (train_transe_on_subgraph tracks them itself)
    # -- no need to re-run evaluate_link_prediction here.
    print(f"[{SCHEMA_LABEL}] emb_dim={dim} | Val MRR pooled={val_mrr_pooled:.4f} "
          f"macro={val_mrr_macro:.4f} (monitor={MONITOR}, selected={val_mrr_selected:.4f}) | {elapsed:.1f}s")

    search_rows.append({
        'model': 'TransE', 'schema': SCHEMA_LABEL, 'phase': 'search', 'emb_dim': dim,
        'val_mrr': val_mrr_pooled, 'val_mrr_macro': val_mrr_macro,
        # best_epoch_this_dim is the epoch the RESTORED checkpoint actually
        # came from -- NOT train_history[-1]['epoch'] (the epoch training
        # stopped at), which is `patience` epochs LATER whenever early
        # stopping genuinely triggered. An earlier version of this cell used
        # the latter and silently mislabeled every run that early-stopped
        # (e.g. reported best_epoch=25 for a checkpoint that actually came
        # from epoch 10) -- see train_transe_on_subgraph's docstring.
        'best_epoch': best_epoch_this_dim,
        'runtime_sec': elapsed, 'status': 'ok',
    })

    if val_mrr_selected > best_val_mrr:
        best_val_mrr, best_dim, best_history = val_mrr_selected, dim, train_history
        if best_model is not None:
            best_model.cpu()
            del best_model
        best_model = model
    else:
        model.cpu()
        del model
    cleanup()

pd.DataFrame(search_rows).to_csv(f"{LP_DIR}transe_{SCHEMA_LABEL}_lp_dimsearch_tuning_results.csv", index=False)
print(f"\n>>> Best emb_dim for TransE/{SCHEMA_LABEL}: {best_dim} "
      f"(val MRR [{MONITOR}]: {best_val_mrr:.4f})")

# Per-epoch training history for the winning emb_dim only -- same convention
# as link_prediction_tools.run_small_lp_validation_search's own history CSV
# for RGCN/GraphSAGE (hidden_dim/emb_dim inserted as the first column so
# this file is self-labeled without cross-referencing the tuning CSV).
dim_history_path = f"{LP_DIR}transe_{SCHEMA_LABEL}_lp_dimsearch_training_history.csv"
history_df = pd.DataFrame(best_history)
history_df.insert(0, 'emb_dim', best_dim)
history_df.to_csv(dim_history_path, index=False)
print(f"Saved per-epoch training history for the winning emb_dim={best_dim} (dim search only, superseded below): {dim_history_path}")

model = best_model
test_score_fn = make_transe_score_fn(model)
test_metrics = evaluate_link_prediction(
    test_score_fn, lp_data['test_triples_by_relation'], lp_data['candidate_pools'],
    lp_data['true_tails_by_relation'], pykeen_relation_fwd_ids,
)

result_rows = []
for relation_name, m in test_metrics.items():
    # emb_dim recorded explicitly, same convention as the RGCN/GraphSAGE
    # results CSVs' hidden_dim column -- this reflects the ONE winning dim
    # from the search above, not every dim tried
    row = {'model': 'TransE', 'schema': SCHEMA_LABEL, 'emb_dim': best_dim, 'relation': relation_name}
    if relation_name in lp_data['train_triples_by_relation']:
        row['num_train_triples'] = int(lp_data['train_triples_by_relation'][relation_name].size(0))
        row['num_val_triples'] = int(lp_data['val_triples_by_relation'][relation_name].size(0))
    row.update(m)
    result_rows.append(row)
results_df = pd.DataFrame(result_rows)
dim_results_path = f"{LP_DIR}transe_{SCHEMA_LABEL}_lp_dimsearch_results.csv"
results_df.to_csv(dim_results_path, index=False)
print(f"\nSaved (dim search only, superseded by the neg_per_pos step below): {dim_results_path}")
print(results_df.to_string(index=False))

del model, best_model
cleanup()

## Sequential LP tuning extension: neg_per_pos

Sweeps `neg_per_pos` with `emb_dim` locked at `best_dim`. This step's
output is the canonical `_lp_results.csv`/`_lp_training_history.csv`/
`_lp_tuning_results.csv` (the dim-only search above is intermediate).

In [ ]:
NEG_PER_POS_GRID = [1, 4, 10]

neg_search_rows = []
best_neg_val_mrr, best_neg_per_pos, best_neg_model, best_neg_history = -1.0, None, None, None
for neg_per_pos in NEG_PER_POS_GRID:
    print(f"\n[LP SECONDARY SEARCH] TransE/{SCHEMA_LABEL} | neg_per_pos={neg_per_pos} (emb_dim={best_dim} fixed)")
    t0 = time.time()
    model, val_mrr_selected, val_mrr_pooled, val_mrr_macro, best_epoch_this_run, train_history = train_transe_on_subgraph(
        train_factory, emb_dim=best_dim, lr=LP_HPARAMS['lr'], epochs=LP_HPARAMS['epochs'],
        train_triples_by_relation=lp_data['train_triples_by_relation'],
        val_triples_by_relation=lp_data['val_triples_by_relation'],
        candidate_pools=lp_data['candidate_pools'],
        true_tails_by_relation=lp_data['true_tails_by_relation'],
        relation_fwd_ids=pykeen_relation_fwd_ids,
        margin=LP_HPARAMS['margin'], scoring_fct_norm=LP_HPARAMS['scoring_fct_norm'],
        optimizer_cls=LP_HPARAMS['optimizer_cls'], neg_per_pos=neg_per_pos,
        patience=LP_HPARAMS['patience'], eval_frequency=LP_HPARAMS['eval_frequency'], monitor=MONITOR,
        max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
    )
    elapsed = time.time() - t0
    print(f"[{SCHEMA_LABEL}] neg_per_pos={neg_per_pos} | Val MRR pooled={val_mrr_pooled:.4f} "
          f"macro={val_mrr_macro:.4f} (monitor={MONITOR}, selected={val_mrr_selected:.4f}) | {elapsed:.1f}s")

    neg_search_rows.append({
        'model': 'TransE', 'schema': SCHEMA_LABEL, 'phase': 'secondary_neg_per_pos',
        'neg_per_pos': neg_per_pos, 'emb_dim': best_dim,
        'val_mrr': val_mrr_pooled, 'val_mrr_macro': val_mrr_macro,
        'best_epoch': best_epoch_this_run, 'runtime_sec': elapsed, 'status': 'ok',
    })

    if val_mrr_selected > best_neg_val_mrr:
        best_neg_val_mrr, best_neg_per_pos, best_neg_history = val_mrr_selected, neg_per_pos, train_history
        if best_neg_model is not None:
            best_neg_model.cpu()
            del best_neg_model
        best_neg_model = model
    else:
        model.cpu()
        del model
    cleanup()

neg_csv_path = f"{LP_DIR}transe_{SCHEMA_LABEL}_lp_tuning_results.csv"
pd.DataFrame(neg_search_rows).to_csv(neg_csv_path, index=False)
print(f"\n>>> Best neg_per_pos for TransE/{SCHEMA_LABEL}: {best_neg_per_pos} "
      f"(val MRR [{MONITOR}]: {best_neg_val_mrr:.4f})")

history_path = f"{LP_DIR}transe_{SCHEMA_LABEL}_lp_training_history.csv"
history_df = pd.DataFrame(best_neg_history)
history_df.insert(0, 'neg_per_pos', best_neg_per_pos)
history_df.insert(0, 'emb_dim', best_dim)
history_df.to_csv(history_path, index=False)
print(f"Saved per-epoch training history for the winning neg_per_pos={best_neg_per_pos}: {history_path}")

model = best_neg_model
test_score_fn = make_transe_score_fn(model)
test_metrics = evaluate_link_prediction(
    test_score_fn, lp_data['test_triples_by_relation'], lp_data['candidate_pools'],
    lp_data['true_tails_by_relation'], pykeen_relation_fwd_ids,
)

result_rows = []
for relation_name, m in test_metrics.items():
    row = {'model': 'TransE', 'schema': SCHEMA_LABEL, 'emb_dim': best_dim,
           'neg_per_pos': best_neg_per_pos, 'relation': relation_name}
    if relation_name in lp_data['train_triples_by_relation']:
        row['num_train_triples'] = int(lp_data['train_triples_by_relation'][relation_name].size(0))
        row['num_val_triples'] = int(lp_data['val_triples_by_relation'][relation_name].size(0))
    row.update(m)
    result_rows.append(row)
results_df = pd.DataFrame(result_rows)
results_path = f"{LP_DIR}transe_{SCHEMA_LABEL}_lp_results.csv"
results_df.to_csv(results_path, index=False)
print(f"\nSaved final results (emb_dim={best_dim}, neg_per_pos={best_neg_per_pos}): {results_path}")
print(results_df.to_string(index=False))

del model, best_neg_model
cleanup()

## Quick sanity check

In [ ]:
# history_path/results_path are set by the neg_per_pos secondary-search
# cell above (the canonical output of the dim -> neg_per_pos chain).
_ = quick_sanity_check(results_path, history_path, 'TransE', SCHEMA_LABEL, 'emb_dim', best_dim)